In [84]:
tasks = [
    "bace",
    "smol-property_prediction-bbbp",
    "smol-property_prediction-clintox",
    "smol-property_prediction-esol",
    "smol-property_prediction-lipo",
    "smol-property_prediction-hiv",
    "smol-property_prediction-sider",
    "qm9_homo",
    "qm9_lumo",
    "qm9_homo_lumo_gap",
    "chebi-20-text2mol",
    "chebi-20-mol2text",
    "smol-molecule_generation",
    "smol-molecule_captioning",
    "reagent_prediction",
    "forward_reaction_prediction",
    "smol-forward_synthesis",
    "retrosynthesis",
    "smol-retrosynthesis",
]

dump_dir = '/text-mol/Mol-LLM/prediction_dump'

string_only_paths = {}
# classification
string_only_paths['bace'] = '/data/all_checkpoints/bace_mlp_string_only_dump_0318/lightning_logs/version_0'
string_only_paths['bbbp'] = '/data/all_checkpoints/smol-property_prediction-bbbp_mlp_string_only_dump_0318/lightning_logs/version_0'
string_only_paths['clintox'] = '/data/all_checkpoints/smol-property_prediction-clintox_mlp_string_only_dump_0318/lightning_logs/version_0'
string_only_paths['hiv'] = '/data/all_checkpoints/smol-property_prediction-hiv_mlp_string_only_dump_0318/lightning_logs/version_0'
string_only_paths['sider'] = '/data/all_checkpoints/smol-property_prediction-sider_mlp_string_only_dump_0318/lightning_logs/version_0'

# regression
string_only_paths['esol'] = '/data/all_checkpoints/smol-property_prediction-esol_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['lipo'] = '/data/all_checkpoints/smol-property_prediction-lipo_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['qm9_homo'] = '/data/all_checkpoints/qm9_homo_mlp_string_only_12ep_0318/lightning_logs/version_0'

# captioning
string_only_paths['chebi-20-mol2text'] = '/data/all_checkpoints/chebi-20-mol2text_mlp_string_only_12ep_0318/lightning_logs/version_0'

# reaction
string_only_paths['forward'] = '/data/all_checkpoints/forward_reaction_prediction_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['reagent'] = '/data/all_checkpoints/reagent_prediction_mlp_string_only_12ep_0318/lightning_logs/version_0'


graph_only_paths = {}
# classification
graph_only_paths['bace'] = '/data/all_checkpoints/bace_qformer_graph_only_dump_0318/lightning_logs/version_0'
graph_only_paths['bbbp'] = '/data/all_checkpoints/smol-property_prediction-bbbp_qformer_graph_only_dump_0318/lightning_logs/version_0'
graph_only_paths['clintox'] = '/data/all_checkpoints/smol-property_prediction-clintox_qformer_graph_only_dump_0318/lightning_logs/version_0'
graph_only_paths['hiv'] = '/data/all_checkpoints/smol-property_prediction-hiv_qformer_graph_only_dump_0318/lightning_logs/version_0'
graph_only_paths['sider'] = '/data/all_checkpoints/smol-property_prediction-sider_qformer_graph_only_dump_0318/lightning_logs/version_0'

# regression
graph_only_paths['esol'] = '/data/all_checkpoints/smol-property_prediction-esol_qformer_graph_only_12ep_0318/lightning_logs/version_0'
graph_only_paths['lipo'] = '/data/all_checkpoints/smol-property_prediction-lipo_qformer_graph_only_12ep_0318/lightning_logs/version_0'
graph_only_paths['qm9_homo'] = '/data/all_checkpoints/qm9_homo_qformer_graph_only_12ep_0318/lightning_logs/version_0'

In [85]:
import os
import json
import re
import selfies as sf
from rdkit import Chem

def get_prediction_dump(path):
    prediction_files = [f for f in os.listdir(path) if (f.startswith('ft-') or f.startswith('test')) and f.endswith('.json')]
    output_files = [f for f in prediction_files if 'output' in f]

    # read the output files
    output_data = []
    for p in output_files:
        with open(os.path.join(path, p), 'r') as f:
            data = json.load(f)
            output_data.extend(data)

    processed_data = []
    truncated_idx = []

    for i in range(len(output_data)):
        instance = output_data[i]
        prediction = instance['prediction']
        target = instance['target']
        prompt = instance['prompt']
        selfies_pattern = r"(?<=<SELFIES>).*(?=</SELFIES>)"
        try:
            if "input_mol_strings" in instance.keys():
                selfies_string = re.search(selfies_pattern, instance["input_mol_strings"]).group().replace(" ", "")
            else:
                selfies_string = re.search(selfies_pattern, prompt).group().replace(" ", "")
            out = {
                'selfies' : selfies_string,
                'prediction': prediction,
                'target': target
            }
            if 'prob' in instance.keys():
                out['prob'] = instance['prob']
            processed_data.append(out)
        except:
            truncated_idx.append(i)
    # print the truncated ratio
    print(f"Truncated ratio: {len(truncated_idx) / len(output_data)}")
    return processed_data

def convert_string2number(text):
    text = text.replace("<FLOAT>", "").replace("</FLOAT>", "")
    text = text.replace("<", "").replace(">", "").replace("|", "").replace(" ", "").replace("/s", "")
    return float(text)

def rank_error_regression(data):
    for i in range(len(data)):
        prediciton = convert_string2number(data[i]['prediction'])
        target = convert_string2number(data[i]['target'])
        data[i]['prediction'] = prediciton
        data[i]['target'] = target
        data[i]['error_score'] = abs(prediciton - target)
    #  sort the data by mae, and add mae rank to the data
    sorted_data = sorted(data, key=lambda x: x['error_score'])
    for i in range(len(sorted_data)):
        sorted_data[i]['error_rank'] = i + 1
    return sorted_data

def rank_error_classification(data):
    for i in range(len(data)):
        prob_posive = data[i]['prob'][1]
        if "true" in data[i]['target'].lower():
            target = 1
        elif "false" in data[i]['target'].lower():
            target = 0
        else:
            raise ValueError("Target is not true or false")
        data[i]['error_score'] = abs(target - prob_posive)
    #  sort the data by mae, and add mae rank to the data
    sorted_data = sorted(data, key=lambda x: x['error_score'])
    for i in range(len(sorted_data)):
        sorted_data[i]['error_rank'] = i + 1
    return sorted_data

In [86]:
string_only_esol_dump = get_prediction_dump(string_only_paths['esol'])
string_only_esol = rank_error_regression(string_only_esol_dump)
with open(os.path.join(dump_dir, 'dump_esol_string_only.json'), 'w') as f:
    json.dump(string_only_esol, f, indent=4)

string_only_lipo_dump = get_prediction_dump(string_only_paths['lipo'])
string_only_lipo = rank_error_regression(string_only_lipo_dump)
with open(os.path.join(dump_dir, 'dump_lipo_string_only.json'), 'w') as f:
    json.dump(string_only_lipo, f, indent=4)


string_only_qm9_homo_dump = get_prediction_dump(string_only_paths['qm9_homo'])
string_only_qm9_homo = rank_error_regression(string_only_qm9_homo_dump)
with open(os.path.join(dump_dir, 'dump_qm9_homo_string_only.json'), 'w') as f:
    json.dump(string_only_qm9_homo, f, indent=4)

Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0


In [87]:
graph_only_esol_dump = get_prediction_dump(graph_only_paths['esol'])
graph_only_esol = rank_error_regression(graph_only_esol_dump)
with open(os.path.join(dump_dir, 'dump_esol_graph_only.json'), 'w') as f:
    json.dump(graph_only_esol, f, indent=4)

graph_only_lipo_dump = get_prediction_dump(graph_only_paths['lipo'])
graph_only_lipo = rank_error_regression(graph_only_lipo_dump)
with open(os.path.join(dump_dir, 'dump_lipo_graph_only.json'), 'w') as f:
    json.dump(graph_only_lipo, f, indent=4)


graph_only_qm9_homo_dump = get_prediction_dump(graph_only_paths['qm9_homo'])
graph_only_qm9_homo = rank_error_regression(graph_only_qm9_homo_dump)
with open(os.path.join(dump_dir, 'dump_qm9_homo_graph_only.json'), 'w') as f:
    json.dump(graph_only_qm9_homo, f, indent=4)

Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0


In [88]:
# complete bace, bbbp, clintox, hiv, sider, chebi-20-mol2text, forward, reagent
#graph_only_bace = get_prediction_dump(graph_only_paths['bace'])


graph_only_hiv_dump = get_prediction_dump(graph_only_paths['hiv'])
graph_only_hiv = rank_error_classification(graph_only_hiv_dump)
with open(os.path.join(dump_dir, 'dump_hiv_graph_only.json'), 'w') as f:
    json.dump(graph_only_hiv, f, indent=4)


graph_only_bbbp_dump = get_prediction_dump(graph_only_paths['bbbp'])
graph_only_bbbp = rank_error_classification(graph_only_bbbp_dump)
with open(os.path.join(dump_dir, 'dump_bbbp_graph_only.json'), 'w') as f:
    json.dump(graph_only_bbbp, f, indent=4)

graph_only_clintox_dump = get_prediction_dump(graph_only_paths['clintox'])
graph_only_clintox = rank_error_classification(graph_only_clintox_dump)
with open(os.path.join(dump_dir, 'dump_clintox_graph_only.json'), 'w') as f:
    json.dump(graph_only_clintox, f, indent=4)

graph_only_sider_dump = get_prediction_dump(graph_only_paths['sider'])
graph_only_sider = rank_error_classification(graph_only_sider_dump)
with open(os.path.join(dump_dir, 'dump_sider_graph_only.json'), 'w') as f:
    json.dump(graph_only_sider, f, indent=4)

Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0


In [89]:
# complete bace, bbbp, clintox, hiv, sider, chebi-20-mol2text, forward, reagent
string_only_bace = get_prediction_dump(string_only_paths['bace'])
string_only_bace = rank_error_classification(string_only_bace)
with open(os.path.join(dump_dir, 'dump_bace_string_only.json'), 'w') as f:
    json.dump(string_only_bace, f, indent=4)

string_only_hiv = get_prediction_dump(string_only_paths['hiv'])
string_only_hiv = rank_error_classification(string_only_hiv)
with open(os.path.join(dump_dir, 'dump_hiv_string_only.json'), 'w') as f:
    json.dump(string_only_hiv, f, indent=4)

string_only_bbbp = get_prediction_dump(string_only_paths['bbbp'])
string_only_bbbp = rank_error_classification(string_only_bbbp)
with open(os.path.join(dump_dir, 'dump_bbbp_string_only.json'), 'w') as f:
    json.dump(string_only_bbbp, f, indent=4)

string_only_clintox = get_prediction_dump(string_only_paths['clintox'])
string_only_clintox = rank_error_classification(string_only_clintox)
with open(os.path.join(dump_dir, 'dump_clintox_string_only.json'), 'w') as f:
    json.dump(string_only_clintox, f, indent=4)

string_only_sider = get_prediction_dump(string_only_paths['sider'])
string_only_sider = rank_error_classification(string_only_sider)
with open(os.path.join(dump_dir, 'dump_sider_string_only.json'), 'w') as f:
    json.dump(string_only_sider, f, indent=4)



Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
